## Part C1.2. Computing Nematic order using TrackMate-extracted spot data
#### The objective of this jupyter notebook is to extract the relative orientation of single cells to compute the nematic order for each condition. The data processing was based on the segmented masks retrieved using Onimpose in Part A. 

#### Subsequently, we applied TrackMate using the same methods described in Part B to analyze the information of cells (spots) in the colony. It is noted that the properties of the mask was set with 0.065 um pixel size and 5-second frame interval. The TrackMate-extracted CSV file for each condition was save and directly perform batch processing, demonstrated here in this code.

In [1]:
#Import essential libraries 
import pandas as pd
import numpy as np
import os

"""
Perform batch processing for multiple CSV files corresponding to different mutants and densities.
This code will finally save the results in a single CSV file that includes 
all the Nematic order values for all 9 conditions 
"""
# Define the directory where the CSV files are located. It's noted that this depends on the data storage of the user
data_directory = '/Users/herbert/Desktop/ME480/Projects/Project1/Data/Trackmate_output/All'

# Obtain a list of all CSV files in the directory using os.listdir()
file_names = [f for f in os.listdir(data_directory) if f.endswith ('csv')]

# Initialize an empty list to hold the results
all_results = []

### C1.2.1. Calculate the nematic order of the whole region for each frame: 
##### To calculate the nematic order for a given list of Ellipse Theta(in radians). 

##### :param orientations: Array of Ellipse Theta (in radians) for a single frame
##### :return: Nematic order value, which is defined based on the average of the second 
##### Legendre polynomial: Sr_1 = <P(cos(theta)> = <1/2*(3cos^2(theta)-1)>

In [15]:
#Define another function to calculate the Nematic order
def calculate_nematic_order (orientations):
    #Compute the nematic order using the formula described above
    cos_theta = np.cos(orientations)
    nematic_order = np.mean ((3 * cos_theta**2 -1)/2)
    return nematic_order

In [16]:
# Group the data by time and calculate nematic order for each time frame
nematic_order_per_timeframe = []

# Loop over each CSV file, process it and calcualate its Nematic Order
for file_name in file_names:
    file_path = os.path.join(data_directory, file_name)
    data = pd.read_csv(file_path, skiprows = 3) # Read the CSV and skip the first three rows that do not contain numeric numbers

    # Process each Time frame in the CSV file 
    #Group data by Time (sec) and apply the calculation for each group
    for timeframe, group in data.groupby('(sec)'): #The units for the columne of Time 
        #Extract the 'Ellipse angle' values for each time frame 
        theta_values = group ['(radians)'].values #The units for the columne of Ellipse angle
        nematic_order = calculate_nematic_order (theta_values)

        #Append the reulst with condition name, Time , and Nematic order
        condition_name = file_name.split('.')[0] #Extract the condition name from the file name
        # Store the result as a tuple (position_T (time), nematic order)    
        all_results.append((condition_name, timeframe, nematic_order))
    
# Convert the results to a pandas DataFrame for saving
final_df = pd.DataFrame (all_results, columns=['Condition', 'Timeframe', 'Nematic Order'])

# Demonstrate the results from calculation (optional)
# print("Sr_calculated as", all_results)

# Save the results to a new CSV file
output_file = 'all_nematic_order_output.csv' #The file name should be changed based on the experimental dataset 
final_df.to_csv(output_file, index=False)

print("Nematic order for all conditions calculated and saved to", output_file)

Nematic order for all conditions calculated and saved to all_nematic_order_output.csv


### C.1.2.2. Calculate the Nematic Order of the Region of Interest (dense region)

#### From the previous calculation (C1.2.1.), we noticed that the calculated Numatic order is independent to frame. Ideally, we could reduce the frame number to save computational power (optional). Additionally, from what we have learned in the leacture, calculating Numatic order should be defined to a certain region that only considers such order parameter among the neighbouring bacteria. This part aims to first define a certain region that in a relatively close proximity and calculate the Numatic order